In [ ]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.utils import get_column_letter
from openpyxl.styles import Alignment

# Load raw CSV
df = pd.read_csv(r"C:\mydata\G8Vitamin\src\results\24082025_clean_data\metrics_detailed.csv")

# Add run index if missing
if "Run" not in df.columns:
    df["Run"] = range(1, len(df) + 1)

# Extra metrics from confusion matrix
df["F1_label1"] = 2 * (df["PPV"] * df["SEN"]) / (df["PPV"] + df["SEN"])
df["F1_label0"] = 2 * (df["NPV"] * df["SPE"]) / (df["NPV"] + df["SPE"])

metrics = [
    "Training time", "Test time", "ACC", "ACC balanced", "MCC", "CohenKappa",
    "F1_macro", "F1_weighted", "ROC_AUC", "PR_AUC",
    "TP", "FP", "FN", "TN", "PPV", "NPV", "SEN", "SPE",
    "F1_label0", "F1_label1"
]

# Write Excel with one sheet per metric
with pd.ExcelWriter("results_formatted.xlsx", engine="openpyxl") as writer:
    for metric in metrics:
        pivot = df.pivot_table(index="Run", columns="Model", values=metric)
        pivot.reset_index(inplace=True)

        # Write to Excel
        pivot.to_excel(writer, sheet_name=metric, index=False)

    writer.close()

# ---- Post-process: merge top row for each metric ----
wb = load_workbook("results_formatted.xlsx")
for metric in metrics:
    ws = wb[metric]

    # Insert a new row at the top
    ws.insert_rows(1)

    # Merge cells across all model columns for the title
    max_col = ws.max_column
    ws.merge_cells(start_row=1, start_column=2, end_row=1, end_column=max_col)

    # Set metric name in merged cell
    ws.cell(row=1, column=2).value = metric.upper()
    ws.cell(row=1, column=2).alignment = Alignment(horizontal="center", vertical="center")

wb.save("results_formatted.xlsx")


In [2]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Alignment

# Load your input table
df = pd.read_csv(r"C:\mydata\G8Vitamin\src\results\24082025_clean_data\metrics_detailed.csv")


In [ ]:
import pandas as pd
import numpy as np
import os
# Load your input table
df = pd.read_csv(r"C:\mydata\G8Vitamin\src\results\06092025_clean_data\metrics_detailed.csv")
ROOT_DIR = r"C:\mydata\G8Vitamin\src\final_results"  
FOLDERNAME = "06092025_clean_data"
STORE_PATH = os.path.join(ROOT_DIR, FOLDERNAME)

# Create directory if not exists
os.makedirs(STORE_PATH, exist_ok=True)
# --- Add F1 per class ---
def safe_div(a, b):
    return a / b if b != 0 else np.nan

# F1 for label 1
df["F1_label1"] = df.apply(
    lambda row: safe_div(
        2 * (safe_div(row["TP"], row["TP"] + row["FP"])) *
        (safe_div(row["TP"], row["TP"] + row["FN"])),
        (safe_div(row["TP"], row["TP"] + row["FP"])) + (safe_div(row["TP"], row["TP"] + row["FN"]))
    ), axis=1
)

# F1 for label 0
df["F1_label0"] = df.apply(
    lambda row: safe_div(
        2 * (safe_div(row["TN"], row["TN"] + row["FN"])) *
        (safe_div(row["TN"], row["TN"] + row["FP"])),
        (safe_div(row["TN"], row["TN"] + row["FN"])) + (safe_div(row["TN"], row["TN"] + row["FP"]))
    ), axis=1
)

# --- List of metrics to export (including F1 per class) ---
metrics = [
    "Training time", "Test time", "ACC", "ACC balanced", "MCC", "CohenKappa",
    "F1_macro", "F1_weighted", "ROC_AUC", "PR_AUC",
    "TP", "FP", "FN", "TN", "PPV", "NPV", "SEN", "SPE",
    "F1_label0", "F1_label1"
]

# --- Export each metric to its own CSV ---
for metric in metrics:
    pivot = df.pivot_table(columns="Model", values=metric, aggfunc=lambda x: list(x))
    pivot = pivot.apply(lambda col: pd.Series(sum(col.tolist(), [])), axis=0)
    pivot.reset_index(drop=True, inplace=True)

    # Add run index
    pivot.insert(0, "Run", range(1, len(pivot) + 1))

    # Save to CSV
    filename = f"{metric.replace(' ', '_')}.csv"
    excel_path = os.path.join(STORE_PATH, filename)
    pivot.to_csv(excel_path, index=False)
    print(f"Saved {filename}")


PermissionError: [Errno 13] Permission denied: 'C:\\mydata\\G8Vitamin\\src\\results\\06092025_clean_data'

In [4]:
import pandas as pd

# Load your input table
df = pd.read_csv(r"C:\mydata\G8Vitamin\src\results\24082025_clean_data\metrics_detailed.csv")

# List of metrics to export
metrics = [
    "Training time", "Test time", "ACC", "ACC balanced", "MCC", "CohenKappa",
    "F1_macro", "F1_weighted", "ROC_AUC", "PR_AUC",
    "TP", "FP", "FN", "TN", "PPV", "NPV", "SEN", "SPE"
]

for metric in metrics:
    # Pivot so that each model is a column, runs are rows
    pivot = df.pivot_table(columns="Model", values=metric, aggfunc=lambda x: list(x))
    pivot = pivot.apply(lambda col: pd.Series(sum(col.tolist(), [])), axis=0)
    pivot.reset_index(drop=True, inplace=True)

    # Add run index
    pivot.insert(0, "Run", range(1, len(pivot) + 1))

    # Save each metric to CSV
    filename = f"{metric.replace(' ', '_')}.csv"
    pivot.to_csv(filename, index=False)
    print(f"Saved {filename}")


Saved Training_time.csv
Saved Test_time.csv
Saved ACC.csv
Saved ACC_balanced.csv
Saved MCC.csv
Saved CohenKappa.csv
Saved F1_macro.csv
Saved F1_weighted.csv
Saved ROC_AUC.csv
Saved PR_AUC.csv
Saved TP.csv
Saved FP.csv
Saved FN.csv
Saved TN.csv
Saved PPV.csv
Saved NPV.csv
Saved SEN.csv
Saved SPE.csv
